# 02 · DPO pairs from the model's own mistakes

**Decision this notebook supports:** DPO only teaches something when the "rejected" side is a mistake the model would really make. v1's rejected answers were absurd ("Around 18,000 units were delivered"), so the model already avoided them, and DPO changed 2 of 280 answers.

Here: run SFT v2 on 600 fresh prompts in the serving format, collect its real failures, have Gemini confirm them, and pair each with the correct response.

| prompt type | the model fails when it... | chosen |
|---|---|---|
| answer with gold context (look-alike chunks present) | gives another model's or city's value, refuses, or adds an invented caveat | the correct fact sentence |
| near-miss unanswerable | invents an answer, or refuses but adds an invented detail | a clean refusal |
| gold chunk removed | states a **wrong** value (a correct memorised value is not punished) | a clean refusal |

In [ ]:
# Find the v2 folder, put it (and the main lab one folder up) on the import path,
# and apply the memory-safety settings BEFORE torch is imported.
import sys, json, time, re, math, random, hashlib, collections
from pathlib import Path
V2_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/v2.yaml").is_file())
sys.path.insert(0, str(V2_ROOT))
from v2lib import setup, device as D, prompts as P, facts as F
CFG = setup.cfg()
SNAP = setup.read_json(V2_ROOT / CFG["retrieval"]["snapshot"])
CHUNKS = SNAP["chunks"]
print("v2 folder:", V2_ROOT)
print("main lab:", setup.PARENT)
print("device:", D.device(), "| dtype:", D.dtype(), "| base model:", D.base_model_path())
D.memory_report("start")

In [ ]:
prompts = setup.read_jsonl(V2_ROOT / CFG["data"]["dpo_prompts"])
print(len(prompts), "prompts |", dict(collections.Counter(p["kind"] for p in prompts)), "|", dict(collections.Counter(p["split"] for p in prompts)))

## Load SFT v2 (base + adapter)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
DEVICE, DTYPE = D.device(), D.dtype()
tok = AutoTokenizer.from_pretrained(D.base_model_path())
base = AutoModelForCausalLM.from_pretrained(D.base_model_path(), dtype=DTYPE).to(DEVICE)
model = PeftModel.from_pretrained(base, str(V2_ROOT / CFG["sft"]["output_adapter"])).to(DEVICE).eval()
D.memory_report("SFT v2 loaded")

## Generate: one greedy answer + three sampled answers per prompt

Greedy shows what the model does by default. Sampling at temperature 0.8 shows which mistakes are *nearby*, which DPO can push down. Results are saved after every prompt, so an interrupted run resumes where it stopped.

In [ ]:
GEN_PATH = V2_ROOT / "runs/dpo_pairs/generations.jsonl"
GEN_PATH.parent.mkdir(parents=True, exist_ok=True)
done = {r["id"]: r for r in setup.read_jsonl(GEN_PATH)} if GEN_PATH.exists() else {}
d = CFG["dpo"]

def prompt_ids(p):
    return tok.apply_chat_template(P.messages(p["question"], [CHUNKS[c] for c in p["chunk_ids"]]), tokenize=True,
                                   add_generation_prompt=True, return_dict=False)

t0 = time.time()
with GEN_PATH.open("a") as fh:
    for i, p in enumerate(prompts):
        if p["id"] in done:
            continue
        x = torch.tensor([prompt_ids(p)], device=DEVICE)
        with torch.inference_mode():
            g = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=CFG["generation"]["max_new_tokens"],
                               do_sample=False, num_beams=1, pad_token_id=tok.pad_token_id)
            torch.manual_seed(CFG["seed"] + i)
            s = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=CFG["generation"]["max_new_tokens"],
                               do_sample=True, temperature=d["sample_temperature"], top_p=1.0,
                               num_return_sequences=d["samples_per_prompt"], pad_token_id=tok.pad_token_id)
        outs = [tok.decode(g[0, x.shape[1]:], skip_special_tokens=True).strip()] + \
               [tok.decode(s[j, x.shape[1]:], skip_special_tokens=True).strip() for j in range(s.shape[0])]
        row = {"id": p["id"], "answers": outs, "sources": ["greedy"] + ["sample"] * (len(outs) - 1)}
        fh.write(json.dumps(row, ensure_ascii=False) + "\n"); fh.flush()
        done[p["id"]] = row
        if len(done) % 25 == 0:
            setup.progress("02_dpo_pairs", f"generated {len(done)}/{len(prompts)} prompts | {(time.time()-t0)/60:.1f} min")
        if len(done) % 100 == 0:
            print(f"{len(done)}/{len(prompts)} prompts | {(time.time()-t0)/60:.1f} min"); D.memory_report()
print(f"generations ready for {len(done)} prompts")

## Round 2 · sample more widely

SFT v2 is already right on most of these prompts, so round 1 alone yields few real failures. Round 2 draws four more answers per prompt at temperature 1.0. They are still the model's **own** answers, just less conservative, and they surface the mistakes it is close to making. When a prompt has several failures, the pair builder prefers greedy over T=0.8 over T=1.0.

In [ ]:
R2_PATH = V2_ROOT / "runs/dpo_pairs/generations_round2.jsonl"
done2 = {r["id"]: r for r in setup.read_jsonl(R2_PATH)} if R2_PATH.exists() else {}
t0 = time.time()
with R2_PATH.open("a") as fh:
    for i, p in enumerate(prompts):
        if p["id"] in done2:
            continue
        x = torch.tensor([prompt_ids(p)], device=DEVICE)
        torch.manual_seed(10_000 + CFG["seed"] + i)
        with torch.inference_mode():
            s = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=CFG["generation"]["max_new_tokens"],
                               do_sample=True, temperature=d["round2_temperature"], top_p=1.0,
                               num_return_sequences=d["round2_samples"], pad_token_id=tok.pad_token_id)
        outs = [tok.decode(s[j, x.shape[1]:], skip_special_tokens=True).strip() for j in range(s.shape[0])]
        row = {"id": p["id"], "answers": outs, "sources": ["sample_t1.0"] * len(outs)}
        fh.write(json.dumps(row, ensure_ascii=False) + "\n"); fh.flush()
        done2[p["id"]] = row
        if len(done2) % 50 == 0:
            setup.progress("02_dpo_pairs", f"round 2: {len(done2)}/{len(prompts)} prompts | {(time.time()-t0)/60:.1f} min")
for pid, row in done2.items():
    if pid in done and len(done[pid]["answers"]) == 1 + d["samples_per_prompt"]:
        done[pid] = {"id": pid, "answers": done[pid]["answers"] + row["answers"], "sources": done[pid]["sources"] + row["sources"]}
print(f"round 2 ready for {len(done2)} prompts; answers per prompt now {len(next(iter(done.values()))['answers'])}")

## First pass: deterministic checks

Fast rules decide which answers *might* be failures:
- answerable: does the answer contain the gold value? Does it refuse? Does it add a "documents don't specify" hedge?
- unanswerable: does it refuse? Does it state numbers that are not in the question?

Only the suspected failures go to Gemini.

In [ ]:
from src import lab
REFUSAL = re.compile(r"i don't have|i do not have|not in (the|meridian's)( official)? documents|isn't listed|not listed|don't cover|not specified|no information", re.I)
HEDGE = re.compile(r"(don't|doesn't|do not|does not) (specify|provide|list|mention)|not specified|isn't specified", re.I)
PIDX = {p["id"]: p for p in prompts}

def nums(text):
    return F.numbers_in(re.sub(r"(Arc|Pulse) (110|125)", "", text))

def has_value(answer, fact):
    if isinstance(fact["value"], str):
        return all(c.strip().lower() in answer.lower() for c in fact["value"].split(","))
    return lab.contains_value(answer, fact["value"], fact.get("unit"))

def first_pass(p, a):
    a2 = a.replace("’", "'")
    refused = bool(REFUSAL.search(a2))
    if p["kind"] in ("answer_with_gold_context", "refuse_gold_removed"):
        f = F.FIDX[p["gold_fact_ids"][0]]
        gold = has_value(a2, f)
        if p["kind"] == "refuse_gold_removed":
            return "ok" if refused or gold else "suspect_wrong_value"
        if gold and not refused:
            return "suspect_caveat" if HEDGE.search(a2) else "ok"
        return "suspect_over_refusal" if refused and not gold else "suspect_wrong_value"
    extra = [n for n in nums(a2) if n not in nums(p["question"])]
    if refused and not extra:
        return "ok"
    return "suspect_refused_with_invented" if refused else "suspect_invented"

checks = []
for pid, row in done.items():
    for a, src in zip(row["answers"], row["sources"]):
        checks.append({"id": pid, "answer": a, "source": src, "first_pass": first_pass(PIDX[pid], a)})
cdf = __import__("pandas").DataFrame(checks)
display(cdf.groupby(["first_pass", "source"]).size().unstack(fill_value=0))

## Second pass: Gemini confirms each suspected failure

The judge is first checked on 60 answers with known labels; it must agree on at least 90% or the notebook stops. Identical answers to the same prompt are judged once.

In [ ]:
from v2lib import judge as J, gemini_io as G
cal = setup.read_json(V2_ROOT / CFG["data"]["judge_calibration"])
cal_res = J.judge(cal, purpose="judge_calibration", cap=CFG["judge"]["max_api_calls"]["dpo_pairs"])
agree = sum(cal_res[it["id"]]["label"] == it["expected"] for it in cal) / len(cal)
print(f"judge calibration agreement: {agree:.1%}")
assert agree >= CFG["judge"]["calibration_min_agreement"], "judge failed calibration: stop and inspect"

suspects = {}
for c in checks:
    if c["first_pass"] != "ok":
        suspects.setdefault((c["id"], c["answer"]), c)
items = []
for (pid, a), c in suspects.items():
    p = PIDX[pid]
    answerable = p["kind"] in ("answer_with_gold_context", "refuse_gold_removed")
    items.append({"id": hashlib.sha256((pid + a).encode()).hexdigest()[:16], "pid": pid, "question": p["question"],
                  "answerable": answerable, "gold": J.gold_for(F.FIDX[p["gold_fact_ids"][0]]) if answerable else None,
                  "answer": a, "truncated": False})
print(len(items), "distinct suspected failures to confirm")
setup.progress("02_dpo_pairs", f"judging {len(items)} suspected failures")
verdicts = J.judge(items, purpose="dpo_failure_check", cap=CFG["judge"]["max_api_calls"]["dpo_pairs"])
print(G.stats())

## Build the pairs

A failure is **confirmed** only when Gemini agrees: wrong value, invented answer, refusal with an invented fact, or an unsupported caveat. For answerable prompts, an over-refusal is also a failure, because the answer was in the documents. For gold-removed prompts, only a wrong value counts.

One pair per prompt, preferring the greedy answer's failure (the model's default behaviour).

In [ ]:
FAIL = {"answer_with_gold_context": {"wrong_value", "correct_with_unsupported_caveat", "refused", "off_topic"},
        "refuse_gold_removed": {"wrong_value"},
        "refuse_near_miss": {"invented_answer", "refused_with_invented_fact", "off_topic"}}
confirmed = collections.defaultdict(list)
for it in items:
    v = verdicts[it["id"]]
    p = PIDX[it["pid"]]
    if v["label"] in FAIL[p["kind"]]:
        src = suspects[(it["pid"], it["answer"])]["source"]
        confirmed[it["pid"]].append({"answer": it["answer"], "label": v["label"], "source": src, "rationale": v["rationale"]})

pairs = []
for pid, fails in confirmed.items():
    p = PIDX[pid]
    best = sorted(fails, key=lambda f: {"greedy": 0, "sample": 1}.get(f["source"], 2))[0]
    if best["answer"].strip() == p["answer"].strip():
        continue
    pairs.append({"id": pid, "kind": p["kind"], "question": p["question"], "chunk_ids": p["chunk_ids"], "split": p["split"],
                  "chosen": p["answer"], "rejected": best["answer"], "failure": best["label"], "rejected_source": best["source"]})
real = len(pairs)
print(f"real-failure pairs: {real} | prompts with no confirmed failure: {len(prompts) - len(confirmed)}")
print(dict(collections.Counter(p["failure"] for p in pairs)))

## Top up only if needed: plausible look-alike rejects

If fewer than the configured minimum of real failures exist, add rejects that use **another model's or city's real value** (the binding mistake seen in v1), capped at 30% of the pairs and flagged `synthetic_lookalike`. No absurd or off-topic rejects.

In [ ]:
need = max(0, d["min_real_pairs"] - real)
share = d["max_synthetic_share"]
cap = int(real * share / (1 - share))          # synthetic / (real + synthetic) <= share
added = 0
used = {p["id"] for p in pairs}
for p in prompts:
    if added >= min(need, cap):
        break
    if p["id"] in used or p["kind"] != "answer_with_gold_context":
        continue
    f = F.FIDX[p["gold_fact_ids"][0]]
    others = [g for g in F.FACTS if g["attribute"] == f["attribute"] and g["value"] != f["value"]]
    if not others or isinstance(f["value"], str) or f["id"] == "110_fast_charge_minutes":
        continue   # the Arc fast-charge sentence never states a number, so a value swap would not change it
    fake = dict(f, value=random.Random(p["id"]).choice(others)["value"])
    if F.answer_sentence(fake, 0).strip() == p["answer"].strip():
        continue
    pairs.append({"id": p["id"], "kind": p["kind"], "question": p["question"], "chunk_ids": p["chunk_ids"], "split": p["split"],
                  "chosen": p["answer"], "rejected": F.answer_sentence(fake, 0), "failure": "wrong_value", "rejected_source": "synthetic_lookalike"})
    added += 1
print(f"synthetic look-alike pairs added: {added} (cap {cap}; share of final set {added / max(1, real + added):.0%})")

## Checks, then save

In [ ]:
for p in pairs:
    assert p["chosen"].strip() != p["rejected"].strip()
    assert not re.search(r"(?<![\d,.])0 minutes", p["chosen"])
train_pairs = [p for p in pairs if p["split"] == "train"]
val_pairs = [p for p in pairs if p["split"] == "val"]
setup.write_jsonl(V2_ROOT / CFG["data"]["dpo_train"], train_pairs)
setup.write_jsonl(V2_ROOT / CFG["data"]["dpo_val"], val_pairs)
summary = {"prompts": len(prompts), "real_pairs": real, "synthetic_pairs": added, "train": len(train_pairs), "val": len(val_pairs),
           "by_failure": dict(collections.Counter(p["failure"] for p in pairs)), "by_kind": dict(collections.Counter(p["kind"] for p in pairs)),
           "by_source": dict(collections.Counter(p["rejected_source"] for p in pairs)),
           "val_by_kind": dict(collections.Counter(p["kind"] for p in val_pairs)), "gemini": G.stats(),
           "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
setup.write_json(V2_ROOT / "runs/dpo_pairs/summary.json", summary)
print(json.dumps(summary, indent=1))

In [ ]:
import pandas as pd
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(pairs).groupby("failure").head(2)[["failure", "question", "chosen", "rejected"]])
del model, base
D.empty_cache(); D.memory_report("released")